# Task 1 — 80/10/10 single-fold model-selection check

Retrospective diagnostic on labeled Train: fit on 80%, select using the first 10%, inspect the second 10% once. All rows have appeared in previous research, so the final split is not independent confirmation. Test is not read.


In [1]:
from pathlib import Path
import sys,json,joblib
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src'/'pipeline'/'save_model.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from src.pipeline.random_seed import set_seed
from src.pipeline.save_model import save_model
from src.pipeline.evaluate_model import evaluate_model
SEED=set_seed()
import numpy as np,pandas as pd
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import fbeta_score,mean_absolute_error
from IPython.display import display
SRC=ROOT/'src'


In [2]:
TASK=SRC/'task1'; SAVE=TASK/'save'
data={}
for name in ['eda_original_v1','eda_combined_v1','eda_error_context_v2']:
 card=json.loads((TASK/'data'/'eda'/f'{name}.json').read_text(encoding='utf-8'))
 data[name]=pd.read_parquet(SRC/card['splits']['train']['path'])
base=data['eda_original_v1']; y=base.is_cheater.astype(int)
for frame in data.values():assert frame.player_id.equals(base.player_id) and frame.is_cheater.equals(base.is_cheater)
all_pos=np.arange(len(y))
fit_pos,temp_pos=train_test_split(all_pos,test_size=.2,stratify=y,random_state=SEED)
select_pos,check_pos=train_test_split(temp_pos,test_size=.5,stratify=y.iloc[temp_pos],random_state=SEED)
assert not(set(fit_pos)&set(select_pos)) and not(set(fit_pos)&set(check_pos)) and not(set(select_pos)&set(check_pos))
assert (len(fit_pos),len(select_pos),len(check_pos))==(72652,9082,9082)
from sklearn.model_selection import FixedThresholdClassifier
from sklearn.frozen import FrozenEstimator
GRID=np.arange(.30,.501,.025)
results={}
def run(name,source,data_name):
 paths=[SAVE/f'{name}.joblib',SAVE/f'{name}.json',SAVE/f'{name}-evaluation.json']
 source_model=joblib.load(SAVE/f'{source}.joblib')
 columns=list(source_model.feature_names_in_)
 X=data[data_name][columns]
 if any(p.exists() for p in paths):
  assert all(p.exists() for p in paths),f'incomplete {name}'
  card=json.loads(paths[1].read_text(encoding='utf-8'))
  assert card['data_name']==data_name and card['card']['source_model']==source
  model=joblib.load(paths[0]); threshold=float(card['card']['selection_threshold'])
 else:
  fitted=clone(source_model.estimator_);fitted.fit(X.iloc[fit_pos],y.iloc[fit_pos])
  prob=fitted.predict_proba(X.iloc[select_pos])[:,1]
  scores=[fbeta_score(y.iloc[select_pos],prob>=v,beta=2,average='macro') for v in GRID]
  threshold=float(GRID[int(np.argmax(scores))])
  model=FixedThresholdClassifier(FrozenEstimator(fitted),threshold=threshold,response_method='predict_proba')
  model.fit(X.iloc[fit_pos],y.iloc[fit_pos])
  save_model(task=1,name=name,model=model,data_name=data_name,
   card={'idea':'80/10/10 retrospective selection check','source_model':source,'seed':SEED,
         'fit_rows':len(fit_pos),'selection_rows':len(select_pos),'check_rows':len(check_pos),
         'selection_threshold':threshold,'threshold_grid':GRID.tolist()})
 selection=fbeta_score(y.iloc[select_pos],model.predict(X.iloc[select_pos]),beta=2,average='macro')
 check_pred=np.asarray(model.predict(X.iloc[check_pos])).reshape(-1).astype(int)
 check=fbeta_score(y.iloc[check_pos],check_pred,beta=2,average='macro')
 if not paths[2].exists():
  evaluate_model(task=1,name=name,score=float(check),metric_name='macro_F2_split801010_check',
   higher_is_better=True,card={'split':'retrospective single-fold 80/10/10 check',
   'selection_macro_F2':float(selection),'selection_threshold':threshold,'seed':SEED})
 else:
  saved=json.loads(paths[2].read_text(encoding='utf-8'));assert abs(saved['score']-check)<1e-10
 results[name]={'selection':selection,'check':check,'threshold':threshold,'pred':check_pred}
 print(name,'selection',round(selection,6),'check',round(check,6),'threshold',threshold)


## Fixed candidate configurations
Each base estimator is cloned and fitted on the 80% split. The 10% selection split chooses one threshold per candidate; the last 10% supplies a diagnostic score.


In [3]:
run('t1_split801010_pruned_v1','t1_lgbm_combined_pruned_cv6_full_v1','eda_combined_v1')


t1_split801010_pruned_v1 selection 0.779724 check 0.772633 threshold 0.37500000000000006


In [4]:
run('t1_split801010_original_v1','t1_lgbm_original_threshold_v1','eda_original_v1')


t1_split801010_original_v1 selection 0.782637 check 0.7726 threshold 0.37500000000000006


In [5]:
run('t1_split801010_shallow_v1','t1_lgbm_shallower_more_trees_threshold_cv6_v3','eda_error_context_v2')


t1_split801010_shallow_v1 selection 0.780168 check 0.77318 threshold 0.35000000000000003


## Selection versus check
The model chosen on the selection split is compared with the check ranking.


In [6]:
table=pd.DataFrame([{**{'name':name},**{k:v for k,v in r.items() if k!='pred'}} for name,r in results.items()])
table['selection_rank']=table.selection.rank(ascending=False,method='min').astype(int)
table['check_rank']=table.check.rank(ascending=False,method='min').astype(int)
display(table.sort_values('selection_rank').round(5))
chosen=table.sort_values('selection_rank').iloc[0]['name']; checked=table.sort_values('check_rank').iloc[0]['name']
print('Selected on 10%:',chosen,'; best on check 10%:',checked,'; selection optimism:',round(table.loc[table.name==chosen,'selection'].iloc[0]-table.loc[table.name==chosen,'check'].iloc[0],6))
for a,b in [('t1_split801010_pruned_v1','t1_split801010_original_v1'),('t1_split801010_pruned_v1','t1_split801010_shallow_v1')]:
 pa,pb=results[a]['pred'],results[b]['pred'];print(a,'vs',b,'check disagreements',int((pa!=pb).sum()),'of',len(pa))


,name,selection,check,threshold,selection_rank,check_rank
1,t1_split801010_original_v1,0.78264,0.77260,0.375,1,3
2,t1_split801010_shallow_v1,0.78017,0.77318,0.350,2,1
0,t1_split801010_pruned_v1,0.77972,0.77263,0.375,3,2


Selected on 10%: t1_split801010_original_v1 ; best on check 10%: t1_split801010_shallow_v1 ; selection optimism: 0.010038
t1_split801010_pruned_v1 vs t1_split801010_original_v1 check disagreements 385 of 9082
t1_split801010_pruned_v1 vs t1_split801010_shallow_v1 check disagreements 398 of 9082


## Assessment

The selection split picks original-feature LightGBM (0.78264 macro F₂), while the check split ranks it third (0.77260) and puts shallow LightGBM first (0.77318). The selection winner's score drops by 0.01004 between the two 10% samples. Differences around 0.0001–0.003 are too small to trust from one split. This retrospective result supports retaining backups, not changing the current full-data submission model.
